In [35]:
%pip install pandas numpy requests clickhouse-connect sqlalchemy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from enum import unique

from numpy.ma.core import append
import requests
import pandas as pd

url = "https://BrsApi.ir/Api/Tsetmc/AllSymbols.php?key=BusnGbvtCSfYqNKW2B5TXFHm96mnT98A"

In [37]:
headers = {

        "User-Agent": "Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36 OPR/106.0.0.0",

        "Accept": "application/json, text/plain, */*"
        }

In [38]:
response = requests.get(url, headers=headers)

In [39]:
if response.status_code == 200:
    try:
        # Parse JSON data
        data = response.json()

        # If the data is a list of records, convert directly
        df = pd.DataFrame(data)
        print(df.head())  # Show first few rows
        # Now you can work with the DataFrame: save to CSV, analyze, etc.
    except ValueError:
        print("Response is not valid JSON")
        print(response.text)
else:
    print(f"Error: {response.status_code}")
    print(response.text)

       time    l18                        l30          isin  \
0  15:39:13  زهلال            کشت و صنعت هلال  IRO3HELZ0001   
1  12:30:00  کولان                   کولان سل  IRO3KOLZ0001   
2  12:30:01  وبملت                   بانک ملت  IRO1BMLT0001   
3  12:30:00  شیفته  کشت و صنعت شیفته آرای شرق  IRO3SFHZ0001   
4  15:46:44   عیار       صندوق طلای عیار مفید  IRTKMOFD0001   

                  id                                        cs  cs_id  \
0  48636523343139021                      زراعت و خدمات وابسته      1   
1  64380036977879368                             محصولات کاغذی     21   
2    778253364357513                  بانک‌ها و موسسات اعتباری     57   
3  20292922178776915  محصولات غذایی و آشامیدنی به جز قند و شکر     42   
4  34144395039913458            صندوق سرمایه‌گذاری قابل معامله     68   

               z  bvol                mv  ...     pd4     po4    qo4  zo4  \
0      500000000     1    18429500000000  ...       0       0      0    0   
1     1020000000     1    31

In [40]:
cols = [
    'time', 'l18', 'tvol', 'tno', 'tval', 'pc', 'pcp', 'plp',
    'Buy_CountI', 'Buy_CountN', 'Sell_CountI', 'Sell_CountN',
    'Buy_I_Volume', 'Buy_N_Volume', 'Sell_I_Volume', 'Sell_N_Volume'
]
df = df[cols]

In [41]:
df['time'] = pd.to_datetime(df['time'])

df['year_month'] = df['time'].dt.to_period('M')
df['monthly_avg_volume'] = df.groupby(['l18', 'year_month'])['tvol'].transform('mean')

df['Buy_AvgI'] = df['Buy_I_Volume'] / df['Buy_CountI'].replace(0, pd.NA)
df['Buy_AvgN'] = df['Buy_N_Volume'] / df['Buy_CountN'].replace(0, pd.NA)
df['Sell_AvgI'] = df['Sell_I_Volume'] / df['Sell_CountI'].replace(0, pd.NA)
df['Sell_AvgN'] = df['Sell_N_Volume'] / df['Sell_CountN'].replace(0, pd.NA)

total_buy = df['Buy_I_Volume'] + df['Buy_N_Volume']
total_sell = df['Sell_I_Volume'] + df['Sell_N_Volume']
df['Buy_PctN'] = (df['Buy_N_Volume'] / total_buy.replace(0, pd.NA)) * 100
df['Buy_PctI'] = (df['Buy_I_Volume'] / total_buy.replace(0, pd.NA)) * 100
df['Sell_PctN'] = (df['Sell_N_Volume'] / total_sell.replace(0, pd.NA)) * 100
df['Sell_PctI'] = (df['Sell_I_Volume'] / total_sell.replace(0, pd.NA)) * 100

# دو خط جدید برای نسبت خریدار به فروشنده
df['Buy_Sell_RatioI'] = df['Buy_I_Volume'] / df['Sell_I_Volume'].replace(0, pd.NA)
df['Buy_Sell_RatioN'] = df['Buy_N_Volume'] / df['Sell_N_Volume'].replace(0, pd.NA)

print(df.head())

                 time    l18         tvol     tno            tval      pc  \
0 2026-06-17 15:39:13  زهلال     39548343  989368   1457712889674   36859   
1 2026-06-17 12:30:00  کولان     50867326  183148   1557813981550   30650   
2 2026-06-17 12:30:01  وبملت  20986257878  146270  29480220986333    1405   
3 2026-06-17 12:30:00  شیفته     64115933  126627   1409352698050   22000   
4 2026-06-17 15:46:44   عیار     80472826   70004  36531243523425  453957   

       pcp      plp  Buy_CountI  Buy_CountN  ...       Buy_AvgI  \
0  3585.90  3551.80     1140019         670  ...      31.773247   
1    -1.61    -2.57        6512          10  ...    6290.954392   
2     0.86     2.94       22723         156  ...  678274.433878   
3    -2.44    -2.22        4835          13  ...   11720.815305   
4     1.95     1.71       26813          54  ...    2829.716928   

          Buy_AvgN       Sell_AvgI      Sell_AvgN   Buy_PctN   Buy_PctI  \
0      12343.91791            <NA>     44492530.0  18.58834

C:\Users\MuhAmmaDrezAkhA\AppData\Local\Temp\ipykernel_3980\2083330567.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['time'] = pd.to_datetime(df['time'])


In [42]:
drop_cols = [
    'Buy_CountI', 'Buy_CountN', 'Sell_CountI', 'Sell_CountN',
    'Buy_I_Volume', 'Buy_N_Volume', 'Sell_I_Volume', 'Sell_N_Volume',
    'year_month'
]
df = df.drop(columns=drop_cols)
print(df.head())

                 time    l18         tvol     tno            tval      pc  \
0 2026-06-17 15:39:13  زهلال     39548343  989368   1457712889674   36859   
1 2026-06-17 12:30:00  کولان     50867326  183148   1557813981550   30650   
2 2026-06-17 12:30:01  وبملت  20986257878  146270  29480220986333    1405   
3 2026-06-17 12:30:00  شیفته     64115933  126627   1409352698050   22000   
4 2026-06-17 15:46:44   عیار     80472826   70004  36531243523425  453957   

       pcp      plp  monthly_avg_volume       Buy_AvgI         Buy_AvgN  \
0  3585.90  3551.80        3.954834e+07      31.773247      12343.91791   
1    -1.61    -2.57        5.086733e+07    6290.954392         990063.1   
2     0.86     2.94        2.098626e+10  678274.433878  35729666.134615   
3    -2.44    -2.22        6.411593e+07   11720.815305    572753.153846   
4     1.95     1.71        8.047283e+07    2829.716928     81931.851852   

        Sell_AvgI      Sell_AvgN   Buy_PctN   Buy_PctI  Sell_PctN  Sell_PctI  \
0     

In [43]:
rename_map = {
    'time': 'تاریخ (time)',
    'l18': 'نماد (l18)',
    'monthly_avg_volume': 'میانگین حجم معاملات (monthly_avg_volume)',
    'tvol': 'حجم معاملات (tvol)',
    'tval': 'ارزش معاملات (tval)',
    'pc': 'قیمت پایانی (pc)',
    'Buy_AvgI': 'سرانه خرید حقیقی (Buy_AvgI)',
    'Buy_AvgN': 'سرانه خرید حقوقی (Buy_AvgN)',
    'Sell_AvgI': 'سرانه فروش حقیقی (Sell_AvgI)',
    'Sell_AvgN': 'سرانه فروش حقوقی (Sell_AvgN)',
    'Buy_Sell_RatioI': 'نسبت حقیقی (Buy_Sell_RatioI)',
    'Buy_Sell_RatioN': 'نسبت حقوقی (Buy_Sell_RatioN)',
    'Buy_PctI': 'درصد خرید حقیقی (Buy_PctI)',
    'Buy_PctN': 'درصد خرید حقوقی (Buy_PctN)',
    'Sell_PctI': 'درصد فروش حقیقی (Sell_PctI)',
    'Sell_PctN': 'درصد فروش حقوقی (Sell_PctN)',
    'pcp': 'درصد پایانی (pcp)',
    'plp': 'درصد آخرین قیمت (plp)'
}

new_order = [
    'تاریخ (time)',
    'نماد (l18)',
    'حجم معاملات (tvol)',
    'میانگین حجم معاملات (monthly_avg_volume)',
    'ارزش معاملات (tval)',
    'قیمت پایانی (pc)',
    'سرانه خرید حقیقی (Buy_AvgI)',
    'سرانه خرید حقوقی (Buy_AvgN)',
    'سرانه فروش حقیقی (Sell_AvgI)',
    'سرانه فروش حقوقی (Sell_AvgN)',
    'نسبت حقیقی (Buy_Sell_RatioI)',
    'نسبت حقوقی (Buy_Sell_RatioN)',
    'درصد خرید حقیقی (Buy_PctI)',
    'درصد خرید حقوقی (Buy_PctN)',
    'درصد فروش حقیقی (Sell_PctI)',
    'درصد فروش حقوقی (Sell_PctN)',
    'درصد پایانی (pcp)',
    'درصد آخرین قیمت (plp)'
]

df = df.rename(columns=rename_map)
df = df[new_order]
print(df.head())

         تاریخ (time) نماد (l18)  حجم معاملات (tvol)  \
0 2026-06-17 15:39:13      زهلال            39548343   
1 2026-06-17 12:30:00      کولان            50867326   
2 2026-06-17 12:30:01      وبملت         20986257878   
3 2026-06-17 12:30:00      شیفته            64115933   
4 2026-06-17 15:46:44       عیار            80472826   

   میانگین حجم معاملات (monthly_avg_volume)  ارزش معاملات (tval)  \
0                              3.954834e+07        1457712889674   
1                              5.086733e+07        1557813981550   
2                              2.098626e+10       29480220986333   
3                              6.411593e+07        1409352698050   
4                              8.047283e+07       36531243523425   

   قیمت پایانی (pc) سرانه خرید حقیقی (Buy_AvgI) سرانه خرید حقوقی (Buy_AvgN)  \
0             36859                   31.773247                 12343.91791   
1             30650                 6290.954392                    990063.1   
2              14

In [44]:
df

,تاریخ (time),نماد (l18),حجم معاملات (tvol),میانگین حجم معاملات (monthly_avg_volume),ارزش معاملات (tval),قیمت پایانی (pc),سرانه خرید حقیقی (Buy_AvgI),سرانه خرید حقوقی (Buy_AvgN),سرانه فروش حقیقی (Sell_AvgI),سرانه فروش حقوقی (Sell_AvgN),نسبت حقیقی (Buy_Sell_RatioI),نسبت حقوقی (Buy_Sell_RatioN),درصد خرید حقیقی (Buy_PctI),درصد خرید حقوقی (Buy_PctN),درصد فروش حقیقی (Sell_PctI),درصد فروش حقوقی (Sell_PctN),درصد پایانی (pcp),درصد آخرین قیمت (plp)
0,2026-06-17 15:39:13,زهلال,39548343,3.954834e+07,1457712889674,36859,31.773247,12343.91791,<NA>,44492530.0,<NA>,0.185883,81.411655,18.588345,0.0,100.0,3585.90,3551.80
1,2026-06-17 12:30:00,کولان,50867326,5.086733e+07,1557813981550,30650,6290.954392,990063.1,289.133216,16498.084906,0.834037,5.661395,80.536364,19.463636,96.562043,3.437957,-1.61,-2.57
2,2026-06-17 12:30:01,وبملت,20986257878,2.098626e+10,29480220986333,1405,678274.433878,35729666.134615,1145036.937338,26381818.16,0.871323,1.690203,73.440582,26.559418,84.286254,15.713746,0.86,2.94
3,2026-06-17 12:30:00,شیفته,64115933,6.411593e+07,1409352698050,22000,11720.815305,572753.153846,522.01828,46028.609375,0.926435,2.527569,88.386988,11.613012,95.405462,4.594538,-2.44,-2.22
4,2026-06-17 15:46:44,عیار,80472826,8.047283e+07,36531243523425,453957,2829.716928,81931.851852,5363.111797,264598.888889,1.149434,0.309645,94.490091,5.509909,82.205752,17.794248,1.95,1.71
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446,2026-06-17 06:10:14,ونیکی3,0,0.000000e+00,0,6060,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00
1447,2026-06-17 06:10:08,وبوعلی3,0,0.000000e+00,0,2346,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00
1448,2026-06-17 06:10:09,قپیرا3,0,0.000000e+00,0,15810,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00
1449,2026-06-17 06:10:14,شلعاب3,0,0.000000e+00,0,1086,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.00,0.00


In [17]:
print(df["cs"].nunique())

KeyError: 'cs'

In [9]:
cs = list(df["cs"].unique())
cs

['صندوق سرمایه\u200cگذاری قابل معامله',
 'محصولات شیمیایی',
 'ماشین\u200cآلات و تجهیزات',
 'فلزات اساسی',
 'اطلاعات و ارتباطات',
 'بیمه و صندوق بازنشستگی به جز تامین اجتماعی',
 'سیمان، آهک و گچ',
 'استخراج کانه\u200cهای فلزی',
 'بانک\u200cها و موسسات اعتباری',
 'حمل و نقل، انبارداری و ارتباطات',
 'انبوه\u200cسازی، املاک و مستغلات',
 'سرمایه\u200cگذاری\u200cها',
 'محصولات غذایی و آشامیدنی به جز قند و شکر',
 'شرکت\u200cهای چند رشته\u200cای صنعتی',
 'خودرو و ساخت قطعات',
 'قند و شکر',
 'ساخت محصولات فلزی',
 'رایانه و فعالیت\u200cهای وابسته به آن',
 'مواد و محصولات دارویی',
 'زراعت و خدمات وابسته',
 'سایر محصولات کانی غیرفلزی',
 'ساخت دستگاه\u200cها و وسایل ارتباطی',
 'سایر واسطه\u200cگری\u200cهای مالی',
 'لاستیک و پلاستیک',
 'فراورده\u200cهای نفتی، کک و سوخت هسته\u200cای',
 'پیمانکاری صنعتی',
 'استخراج زغال سنگ',
 'کاشی و سرامیک',
 'هتل و رستوران',
 'فعالیت\u200cهای کمکی به نهادهای مالی واسط',
 'تولید محصولات کامپیوتری الکترونیکی و نوری',
 'محصولات کاغذی',
 'عرضه برق، گاز، بخار و آب گرم',

In [10]:
df.columns

Index(['time', 'l18', 'l30', 'isin', 'id', 'cs', 'cs_id', 'z', 'bvol', 'mv',
       'eps', 'pe', 'tmin', 'tmax', 'pmin', 'pmax', 'py', 'pf', 'pl', 'plc',
       'plp', 'pc', 'pcc', 'pcp', 'tno', 'tvol', 'tval', 'Buy_CountI',
       'Buy_CountN', 'Sell_CountI', 'Sell_CountN', 'Buy_I_Volume',
       'Buy_N_Volume', 'Sell_I_Volume', 'Sell_N_Volume', 'zd1', 'qd1', 'pd1',
       'po1', 'qo1', 'zo1', 'zd2', 'qd2', 'pd2', 'po2', 'qo2', 'zo2', 'zd3',
       'qd3', 'pd3', 'po3', 'qo3', 'zo3', 'zd4', 'qd4', 'pd4', 'po4', 'qo4',
       'zo4', 'zd5', 'qd5', 'pd5', 'po5', 'qo5', 'zo5'],
      dtype='str')

In [11]:
import clickhouse_connect

# Connection parameters (adjust if needed)
CLICKHOUSE_HOST = 'localhost'
CLICKHOUSE_PORT = 9000
DATABASE = 'IranMarket'

#list of 51 table names
TABLE_NAMES = [
'صندوق سرمایه\u200cگذاری قابل معامله',
 'ماشین\u200cآلات و دستگاه\u200cهای برقی',
 'بیمه و صندوق بازنشستگی به جز تامین اجتماعی',
 'بانک\u200cها و موسسات اعتباری',
 'استخراج کانه\u200cهای فلزی',
 'فلزات اساسی',
 'فراورده\u200cهای نفتی، کک و سوخت هسته\u200cای',
 'خودرو و ساخت قطعات',
 'مخابرات',
 'محصولات شیمیایی',
 'شرکت\u200cهای چند رشته\u200cای صنعتی',
 'انبوه\u200cسازی، املاک و مستغلات',
 'سرمایه\u200cگذاری\u200cها',
 'کاشی و سرامیک',
 'فعالیت\u200cهای کمکی به نهادهای مالی واسط',
 'مواد و محصولات دارویی',
 'حمل و نقل، انبارداری و ارتباطات',
 'محصولات غذایی و آشامیدنی به جز قند و شکر',
 'سایر محصولات کانی غیرفلزی',
 'زراعت و خدمات وابسته',
 'قند و شکر',
 'ساخت محصولات فلزی',
 'سیمان، آهک و گچ',
 'اطلاعات و ارتباطات',
 'تولید محصولات کامپیوتری الکترونیکی و نوری',
 'لاستیک و پلاستیک',
 'ماشین\u200cآلات و تجهیزات',
 'خدمات فنی و مهندسی',
 'عرضه برق، گاز، بخار و آب گرم',
 'سایر واسطه\u200cگری\u200cهای مالی',
 'استخراج نفت گاز و خدمات جنبی جز اکتشاف',
 'خرده\u200cفروشی، به\u200cاستثنای وسایل نقلیه موتوری',
 'استخراج سایر معادن',
 '',
 'هتل و رستوران',
 'استخراج زغال سنگ',
 'رایانه و فعالیت\u200cهای وابسته به آن',
 'پیمانکاری صنعتی',
 'منسوجات',
 'محصولات کاغذی',
 'محصولات چوبی',
 'دباغی، پرداخت چرم و ساخت انواع پاپوش',
 'ساخت دستگاه\u200cها و وسایل ارتباطی',
 'فعالیت\u200cهای فرهنگی و ورزشی',
 'تجارت عمده\u200cفروشی به جز وسایل نقلیه موتوری',
 'انتشار، چاپ و تکثیر',
 'حمل و نقل آبی',
 'فعالیت مهندسی، تجزیه، تحلیل و آزمایش فنی',
 'فعالیت\u200cهای هنری، سرگرمی و خلاقانه'
]
# Base table schema (customise for your data)
# This example includes typical columns for market trade data
BASE_SCHEMA = """
    trade_date Date,
    symbol String,
    open_price Float64,
    high_price Float64,
    low_price Float64,
    close_price Float64,
    volume UInt64,
    value Float64,
    created_at DateTime DEFAULT now()
"""

def create_table_if_not_exists(client, table_name):
    query = f"""
    CREATE TABLE IF NOT EXISTS {IranMarket}.{table_name} (
        {BASE_SCHEMA}
    ) ENGINE = MergeTree()
    ORDER BY (trade_date, symbol)
    """
    client.command(query)
    print(f"✅ Table {table_name} ensured")

def main():
    client = clickhouse_connect.get_client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        username='default',
        password=''
    )

    # Switch to your database (or create it if missing)
    client.command(f"CREATE DATABASE IF NOT EXISTS {IranMarket}")
    client.command(f"USE {IranMarket}")

    for table in TABLE_NAMES:
        create_table_if_not_exists(client, table)

    print(f"All {len(TABLE_NAMES)} tables have been processed.")

# if __name__ == "__main__":
#     main()